# 🔊 Notebook 2：聲紋分析模型訓練

**專案**: AI_Voice 智慧語音詐騙檢測工具  
**目標**:  
- Layer 1：LightGBM 韻律異常偵測（Jitter/Shimmer/HNR/F0/Formant/Pause 20 維）  
- Layer 2：Wav2vec2 + CNN 深偽偵測  
**平台**: Kaggle GPU (T4 x2)  
**輸出**: `prosody_lgbm.pkl` + `deepfake_cnn.pt`

In [ ]:
# === Matplotlib 中文顯示修復 (手動路徑版) ===
!apt-get install -y fonts-wqy-microhei
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# 手動強制加載字型檔，避開快取更新問題
font_path = '/usr/share/fonts/truetype/wqy/wqy-microhei.ttc'
if os.path.exists(font_path):
    fm.fontManager.addfont(font_path)
    plt.rcParams['font.sans-serif'] = ['WenQuanYi Micro Hei']
    plt.rcParams['axes.unicode_minus'] = False
    print('✅ 已手動載入字型: WenQuanYi Micro Hei')
else:
    print('❌ 找不到字型檔，請確認已執行 apt-get install')


In [ ]:
# === 環境自我檢測 ===
import socket, os
def check_internet():
    try:
        socket.create_connection(("huggingface.co", 80), 2)
        return True
    except: return False

print('📡 Internet 狀態:', '✅ 已開啟' if check_internet() else '❌ 未開啟 (右側選單勾選 Internet)')
print('📁 當前路徑:', os.getcwd())
print('📀 GPU 資訊:', os.popen('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader').read().strip())
os.makedirs('output', exist_ok=True)


In [ ]:
# === 步驟 A：安裝依賴 (顯式模式) ===
!pip install praat-parselmouth lightgbm librosa noisereduce
!pip install praat-parselmouth transformers torch torchaudio


In [ ]:
# === 步驟 B：導入與環境初始化 ===
# === 安裝依賴 ===

import os, json, pickle, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import librosa
import parselmouth
from parselmouth.praat import call
import lightgbm as lgb
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import classification_report, roc_auc_score
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'裝置: {device}')

## Part A：LightGBM 韻律異常偵測

使用真人語音 vs AI 合成語音的韻律特徵差異進行分類。

### 資料來源策略
由於 CFAD 資料集需要額外申請，這裡我們使用以下策略：
1. 從公開中文語音資料集（如 Common Voice zh-TW）獲取真人語音
2. 使用 TTS 合成器生成 AI 語音
3. 提取韻律特徵並訓練二元分類器

In [ ]:
def extract_prosody_features(audio, sr=16000):
    """從音頻萃取 20 維韻律特徵向量 (優化穩定版)
    
    支援 Numpy/Torch Tensor (CPU/GPU) 以及單/多聲道輸入。
    """
    try:
        # 1. 前處理：轉換為 Numpy 單聲道
        if hasattr(audio, 'detach'): # Torch Tensor
            audio = audio.detach().cpu().numpy()
        
        audio = np.atleast_1d(audio)
        if audio.ndim > 1:
            if audio.shape[0] < audio.shape[1]:
                audio = np.mean(audio, axis=0)
            else:
                audio = np.mean(audio, axis=1)
        
        # 2. 建立 Praat Sound 物件
        snd = parselmouth.Sound(audio, sampling_frequency=sr)
        
        # 3. Pitch 分析 (F0)
        pitch = call(snd, 'To Pitch', 0.0, 75, 600)
        f0_values = pitch.selected_array['frequency']
        f0_voiced = f0_values[f0_values > 0]
        
        f0_mean = float(np.mean(f0_voiced)) if len(f0_voiced) > 0 else 0.0
        f0_std = float(np.std(f0_voiced)) if len(f0_voiced) > 0 else 0.0
        f0_range = float(np.ptp(f0_voiced)) if len(f0_voiced) > 0 else 0.0
        
        # 4. Jitter / Shimmer / HNR
        point_process = call(snd, 'To PointProcess (periodic, cc)', 75, 600)
        jitter = call(point_process, 'Get jitter (local)', 0, 0, 0.0001, 0.02, 1.3)
        shimmer = call([snd, point_process], 'Get shimmer (local)', 0, 0, 0.0001, 0.02, 1.3, 1.6)
        
        harmonicity = call(snd, 'To Harmonicity (cc)', 0.01, 75, 0.1, 1.0)
        hnr = call(harmonicity, 'Get mean', 0, 0)
        
        # 5. Formant (F1-F4)
        formant = call(snd, 'To Formant (burg)', 0.0, 5, 5500, 0.025, 50)
        formants = []
        for i in range(1, 5):
            try:
                f = call(formant, 'Get mean', i, 0, 0, 'hertz')
                formants.append(f if not np.isnan(f) else 0.0)
            except:
                formants.append(0.0)
        
        # 6. Speaking rate & Pause (基於能量)
        rms = librosa.feature.rms(y=audio, frame_length=512, hop_length=160)[0]
        threshold = np.mean(rms) * 0.5
        voiced_frames = np.sum(rms > threshold)
        total_frames = len(rms)
        speaking_rate = voiced_frames / max(1, total_frames) * (sr / 160)
        
        silent_mask = rms <= threshold
        pause_durations = []
        count = 0
        for is_silent in silent_mask:
            if is_silent:
                count += 1
            elif count > 0:
                pause_durations.append(count * 160 / sr)
                count = 0
        
        pause_mean = float(np.mean(pause_durations)) if pause_durations else 0.0
        pause_std = float(np.std(pause_durations)) if pause_durations else 0.0
        pause_count = len(pause_durations)
        f0_cv = f0_std / f0_mean if f0_mean > 0 else 0.0
        
        # 7. 建立 20 維特徵向量
        raw_vector = [
            jitter, shimmer, hnr,
            f0_mean, f0_std, f0_range,
            speaking_rate,
            pause_mean, pause_std, pause_count,
            *formants, f0_cv,
            jitter * shimmer,
            hnr / max(f0_mean, 1),
            f0_range / max(f0_std, 0.01),
            pause_count / max(len(audio) / sr, 0.1),
            float(np.mean(np.abs(np.diff(f0_voiced)))) if len(f0_voiced) > 1 else 0.0,
        ]
        
        # 8. 清理 NaN / Inf
        return np.nan_to_num(np.array(raw_vector, dtype=np.float32), nan=0.0)
    
    except Exception as e:
        print(f'特徵萃取失敗: {e}')
        return np.zeros(20, dtype=np.float32)


In [ ]:
from tqdm.auto import tqdm
# === 生成訓練資料（合成 vs 真實）===
# 策略：
# 1. 生成不同頻率的正弦波模擬真人語音（含自然抖動）
# 2. 生成乾淨正弦波模擬 AI 合成語音（缺乏自然變動）
# NOTE：正式訓練應使用 CFAD 真實資料集

np.random.seed(42)
N_SAMPLES = 2000  # 每類 1000 個

features_list = []
labels_list = []

for i in tqdm(range(N_SAMPLES), desc="提取韻律特徵"):
    sr = 16000
    duration = np.random.uniform(2.0, 5.0)
    t = np.linspace(0, duration, int(sr * duration))
    
    if i < N_SAMPLES // 2:
        # 真人語音模擬：帶有自然抖動和噪音
        f0 = np.random.uniform(100, 300)
        jitter_amount = np.random.uniform(0.01, 0.04)
        f0_variation = f0 * (1 + jitter_amount * np.random.randn(len(t)))
        phase = np.cumsum(2 * np.pi * f0_variation / sr)
        audio = np.sin(phase)
        # 加入自然噪音和幅度抖動
        audio += np.random.randn(len(t)) * np.random.uniform(0.02, 0.1)
        audio *= (1 + np.random.uniform(0.01, 0.05) * np.random.randn(len(t)))
        # 加入隨機停頓
        n_pauses = np.random.randint(1, 5)
        for _ in range(n_pauses):
            start = np.random.randint(0, len(audio) - sr // 4)
            length = np.random.randint(sr // 10, sr // 3)
            audio[start:start+length] *= 0.02
        label = 0  # 真人
    else:
        # AI 合成語音模擬：極為穩定的特徵
        f0 = np.random.uniform(150, 250)
        audio = np.sin(2 * np.pi * f0 * t)
        # 極低的抖動
        audio += np.random.randn(len(t)) * np.random.uniform(0.001, 0.01)
        # 規則性停頓
        pause_interval = int(sr * np.random.uniform(1.0, 1.5))
        for j in range(0, len(audio), pause_interval):
            audio[j:j+int(sr*0.3)] *= 0.01
        label = 1  # 合成
    
    audio = audio.astype(np.float32)
    audio = audio / (np.max(np.abs(audio)) + 1e-8)  # 正規化
    
    feat = extract_prosody_features(audio, sr)
    features_list.append(feat)
    labels_list.append(label)

X = np.array(features_list)
y = np.array(labels_list)

# 清理 NaN
X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

print(f'特徵矩陣: {X.shape}')
print(f'標籤分布: 真人={sum(y==0)}, 合成={sum(y==1)}')

In [ ]:
# === 訓練 LightGBM ===
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

FEATURE_NAMES = [
    'jitter', 'shimmer', 'hnr',
    'f0_mean', 'f0_std', 'f0_range',
    'speaking_rate',
    'pause_mean', 'pause_std', 'pause_count',
    'F1', 'F2', 'F3', 'F4',
    'f0_cv', 'jitter_x_shimmer',
    'hnr_f0_ratio', 'range_std_ratio',
    'pause_density', 'f0_diff_mean'
]

dtrain = lgb.Dataset(X_train, label=y_train, feature_name=FEATURE_NAMES)
dtest = lgb.Dataset(X_test, label=y_test, reference=dtrain)

params = {
    'objective': 'binary',
    'metric': ['binary_logloss', 'auc'],
    'boosting_type': 'gbdt',
    'num_leaves': 31,
    'learning_rate': 0.05,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'verbose': -1,
    'seed': 42,
}

callbacks = [
    lgb.log_evaluation(50),
    lgb.early_stopping(30)
]

model_lgbm = lgb.train(
    params, dtrain,
    num_boost_round=500,
    valid_sets=[dtest],
    callbacks=callbacks
)

# 評估
y_pred_proba = model_lgbm.predict(X_test)
y_pred = (y_pred_proba > 0.5).astype(int)

print('\n=== LightGBM 評估報告 ===')
print(classification_report(y_test, y_pred, target_names=['真人', '合成']))
print(f'AUC: {roc_auc_score(y_test, y_pred_proba):.4f}')

In [ ]:
# === 特徵重要性 ===
importance = model_lgbm.feature_importance(importance_type='gain')
feat_imp = pd.DataFrame({'feature': FEATURE_NAMES, 'importance': importance})
feat_imp = feat_imp.sort_values('importance', ascending=True)

plt.figure(figsize=(10, 8))
plt.barh(feat_imp['feature'], feat_imp['importance'], color='#00f2ff')
plt.xlabel('Importance (Gain)')
plt.title('LightGBM 韻律特徵重要性排名')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150)
plt.show()

In [ ]:
# === 儲存 LightGBM 模型 ===
os.makedirs('output', exist_ok=True)

# 使用 sklearn 的 wrapper 以便 predict_proba
from sklearn.base import BaseEstimator, ClassifierMixin

class LGBMWrapper(BaseEstimator, ClassifierMixin):
    """LightGBM 模型包裝器，提供 predict_proba 介面"""
    def __init__(self, booster):
        self.booster = booster
    
    def predict_proba(self, X):
        p1 = self.booster.predict(X)
        p0 = 1 - p1
        return np.column_stack([p0, p1])
    
    def predict(self, X):
        return (self.booster.predict(X) > 0.5).astype(int)

wrapper = LGBMWrapper(model_lgbm)
with open('output/prosody_lgbm.pkl', 'wb') as f:
    pickle.dump(wrapper, f)

print('LightGBM 模型已儲存: output/prosody_lgbm.pkl')

---
## Part B：Wav2vec2 + CNN 深偽偵測

In [ ]:
# === CNN 深偽偵測模型定義 ===
from transformers import Wav2Vec2Model, Wav2Vec2FeatureExtractor

class DeepfakeCNN(nn.Module):
    """Wav2vec2 特徵 + 3層 CNN 二元分類器
    
    輸入: Wav2vec2 隱藏層表示 (batch, seq_len, 768)
    輸出: 合成語音機率 (batch, 1)
    """
    def __init__(self, input_dim=768, hidden_dim=256):
        super().__init__()
        # 1D 卷積特徵提取
        self.conv_layers = nn.Sequential(
            nn.Conv1d(input_dim, hidden_dim, kernel_size=5, padding=2),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Dropout(0.3),
            
            nn.Conv1d(hidden_dim, hidden_dim, kernel_size=3, padding=1),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Dropout(0.3),
            
            nn.Conv1d(hidden_dim, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),  # 全域平均池化
        )
        
        # 分類頭
        self.classifier = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1),  # 二元輸出
        )
    
    def forward(self, x):
        # x: (batch, seq_len, 768) → (batch, 768, seq_len)
        if x.dim() == 2:
            x = x.unsqueeze(0)
        x = x.permute(0, 2, 1)
        x = self.conv_layers(x)
        x = x.squeeze(-1)
        return self.classifier(x)

cnn_model = DeepfakeCNN().to(device)
print(f'CNN 參數量: {sum(p.numel() for p in cnn_model.parameters()):,}')

In [ ]:
# === 載入 Wav2vec2 特徵提取器 ===
wav2vec2_model = Wav2Vec2Model.from_pretrained('facebook/wav2vec2-base').to(device)
wav2vec2_model.eval()
feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained('facebook/wav2vec2-base')

def extract_wav2vec2_features(audio, sr=16000):
    """提取 Wav2vec2 深度特徵"""
    inputs = feature_extractor(
        audio, sampling_rate=sr, return_tensors='pt', padding=True
    ).to(device)
    with torch.no_grad():
        outputs = wav2vec2_model(**inputs)
    return outputs.last_hidden_state  # (1, seq_len, 768)

print('Wav2vec2 特徵提取器載入完成')

In [ ]:
from tqdm.auto import tqdm
# === CNN 訓練迴圈 ===
# 使用前面生成的合成數據
from torch.utils.data import TensorDataset

# 提取 Wav2vec2 特徵（限制數量以節省時間）
N_CNN = min(500, N_SAMPLES)  # CNN 訓練樣本數
indices = np.random.choice(N_SAMPLES, N_CNN, replace=False)

print(f'正在提取 {N_CNN} 個樣本的 Wav2vec2 特徵...')
w2v_features = []
w2v_labels = []

for idx in tqdm(indices, desc="提取 Wav2vec2 特徵"):
    sr = 16000
    duration = np.random.uniform(2.0, 3.0)
    t = np.linspace(0, duration, int(sr * duration))
    
    if y[idx] == 0:  # 真人
        f0 = np.random.uniform(100, 300)
        audio = np.sin(2 * np.pi * f0 * t + 0.03 * np.random.randn(len(t)))
        audio += np.random.randn(len(t)) * 0.05
    else:  # 合成
        f0 = np.random.uniform(150, 250)
        audio = np.sin(2 * np.pi * f0 * t)
        audio += np.random.randn(len(t)) * 0.005
    
    audio = audio.astype(np.float32)
    audio = audio / (np.max(np.abs(audio)) + 1e-8)
    
    feat = extract_wav2vec2_features(audio, sr)  # (1, T, 768)
    # 截斷或填充至固定長度
    target_len = 50
    if feat.shape[1] > target_len:
        feat = feat[:, :target_len, :]
    elif feat.shape[1] < target_len:
        pad = torch.zeros(1, target_len - feat.shape[1], 768).to(device)
        feat = torch.cat([feat, pad], dim=1)
    
    w2v_features.append(feat.squeeze(0).cpu())
    w2v_labels.append(y[idx])

w2v_X = torch.stack(w2v_features)
w2v_y = torch.tensor(w2v_labels, dtype=torch.float32)
print(f'Wav2vec2 特徵: {w2v_X.shape}')

# 切分
split = int(0.8 * len(w2v_X))
train_X, test_X = w2v_X[:split], w2v_X[split:]
train_y, test_y = w2v_y[:split], w2v_y[split:]

In [ ]:
# === 訓練 CNN ===
optimizer = torch.optim.AdamW(cnn_model.parameters(), lr=1e-3, weight_decay=0.01)
criterion = nn.BCEWithLogitsLoss()
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=30)

EPOCHS = 30
BATCH_SIZE = 32
best_auc = 0

for epoch in range(EPOCHS):
    cnn_model.train()
    total_loss = 0
    
    # Mini-batch
    perm = torch.randperm(len(train_X))
    for i in range(0, len(train_X), BATCH_SIZE):
        batch_idx = perm[i:i+BATCH_SIZE]
        bx = train_X[batch_idx].to(device)
        by = train_y[batch_idx].to(device)
        
        optimizer.zero_grad()
        out = cnn_model(bx).squeeze(-1)
        loss = criterion(out, by)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    scheduler.step()
    
    # 驗證
    cnn_model.eval()
    with torch.no_grad():
        test_out = cnn_model(test_X.to(device)).squeeze(-1)
        test_prob = torch.sigmoid(test_out).cpu().numpy()
        auc = roc_auc_score(test_y.numpy(), test_prob)
    
    if auc > best_auc:
        best_auc = auc
        torch.save(cnn_model.state_dict(), 'output/deepfake_cnn.pt')
    
    if (epoch + 1) % 5 == 0:
        print(f'Epoch {epoch+1}/{EPOCHS} | Loss: {total_loss:.4f} | AUC: {auc:.4f} | Best: {best_auc:.4f}')

print(f'\n訓練完成! Best AUC: {best_auc:.4f}')

In [ ]:
# === 打包下載 ===
!tar -czf voiceprint_models.tar.gz -C output .
print('\n打包完成！請下載 voiceprint_models.tar.gz')
print('放置路徑:')
print('  prosody_lgbm.pkl → AI_Voice/models/voiceprint/prosody_lgbm.pkl')
print('  deepfake_cnn.pt  → AI_Voice/models/voiceprint/deepfake_cnn.pt')